# Drought recovery comparison (by length)

Compare the **5-year recovery** window across drought lengths for **potomac2** and **wolf2**.

Sequence: 40 yr average spinup → drought (`dry`) → 5 yr recovery (`average`).
Recovery is aligned so $t=0$ is drought end / recovery start.

Members (50-year excluded):
- `1_year_drought`, `3_year_drought`, `10_year_drought`
- `short_baseline` for anomalies

Uses **219 h condensed** outputs (`file_locations_219h.json` / `total_storage`, outlet, `wtd`).

Also maps **WTD anomalies** (drought − baseline) at recovery start and end, with stream overlay, and checks landscape covariates for persistent residuals.

In [ ]:
try:
    from drought_ensemble.analysis.paper_figures import utils
    from drought_ensemble.classes import Domain
except ImportError:
    import sys
    from pathlib import Path

    def _find_project_root() -> Path:
        for candidate in [Path.cwd(), *Path.cwd().resolve().parents]:
            if (candidate / "classes" / "Domain.py").is_file():
                return candidate
            nested = candidate / "drought-ensemble"
            if (nested / "classes" / "Domain.py").is_file():
                return nested
        raise ImportError(
            "Install with `pip install -e .` from drought-ensemble/, "
            "or run this notebook from inside the project tree."
        )

    _root = _find_project_root()
    if str(_root) not in sys.path:
        sys.path.insert(0, str(_root))

    from analysis.paper_figures import utils
    from classes import Domain

In [ ]:
from collections import deque
from math import erfc, sqrt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from matplotlib.colors import TwoSlopeNorm
from matplotlib.lines import Line2D

ENSEMBLE = "droughts"
DOMAINS = ["potomac2", "wolf2"]
DOMAIN_LABELS = {"potomac2": "Potomac", "wolf2": "Wolf"}
DROUGHT_LENGTHS = [1, 3, 10]
SPINUP_YEARS = 40
RECOVERY_YEARS = 5
INTERVAL = 219  # condensed product
STEPS_PER_YEAR = utils.ONE_YEAR // INTERVAL
STREAM_FLOW_PERCENTILE = 97.0

PROJECT_ROOT = Path("/glade/derecho/scratch/bwest/drought-ensemble")
FIG_DIR = PROJECT_ROOT / "analysis" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

COLORS = {1: "#E8C39E", 3: "#B86B2B", 10: "#4A2410"}  # light→dark brown (longest = driest)

outlets = {}
for domain_name in DOMAINS:
    domain = Domain(domain_name, full_config_file_path_given=False)
    outlets[domain_name] = (domain.outlet_x, domain.outlet_y)
    print(domain_name, "outlet", outlets[domain_name])


In [ ]:
def read_condensed_series(domain_name, member, start_year=0):
    """Year-by-year total_storage + outlet from 219 h consolidated files."""
    ox, oy = outlets[domain_name]
    files = utils._file_locations(
        ENSEMBLE, member, domain_name, start_year, interval=INTERVAL
    )
    times, storage, outlet = [], [], []
    for year_offset, path in enumerate(files):
        with xr.open_dataset(path) as ds:
            if "total_storage" in ds:
                stor = np.asarray(ds["total_storage"].values, dtype=np.float64)
            else:
                name = (
                    "subsurface_storage"
                    if "subsurface_storage" in ds
                    else "storage"
                )
                stor = (
                    ds[name]
                    .sum(dim=("x", "y", "z"), skipna=True)
                    .values.astype(np.float64)
                )
            flow = np.asarray(
                ds["overland_flow"].isel(x=ox, y=oy).values, dtype=np.float64
            )
        n = stor.shape[0]
        times.append(
            start_year
            + year_offset
            + np.arange(n, dtype=np.float64) / STEPS_PER_YEAR
        )
        storage.append(stor)
        outlet.append(flow)
    return xr.Dataset(
        {
            "storage": ("time", np.concatenate(storage)),
            "outlet_flow": ("time", np.concatenate(outlet)),
        },
        coords={"time": np.concatenate(times)},
    )


def recovery_anomaly(ds, baseline, drought_length):
    """Return (years_into_recovery, Δstorage, Δoutlet) for the recovery window."""
    t0 = SPINUP_YEARS + drought_length
    t1 = t0 + RECOVERY_YEARS
    d_sel = ds.sel(time=slice(t0, t1))
    b_sel = baseline.sel(time=slice(t0, t1))
    n = min(d_sel.sizes["time"], b_sel.sizes["time"])
    t = d_sel.time.values[:n] - t0
    dS = d_sel.storage.values[:n] - b_sel.storage.values[:n]
    dQ = d_sel.outlet_flow.values[:n] - b_sel.outlet_flow.values[:n]
    return t, dS, dQ


series = {}
for domain_name in DOMAINS:
    series[domain_name] = {}
    members = ["short_baseline"] + [
        f"{L}_year_drought" for L in DROUGHT_LENGTHS
    ]
    for member in members:
        print(f"Loading {domain_name}/{member}…", flush=True)
        ds = read_condensed_series(domain_name, member)
        series[domain_name][member] = ds
        print(
            f"  {float(ds.time.min()):.2f}→{float(ds.time.max()):.2f} "
            f"({ds.sizes['time']} steps)",
            flush=True,
        )

## Recovery: totals and anomalies

Time aligned to recovery onset ($t=0$ = end of drought).

One column per domain; panels stacked so they share the recovery-time axis:
total storage → storage anomaly → outlet flow → outlet-flow anomaly.
Y-axis labels (with units) appear once on the left; grey lines on absolute panels are `short_baseline` over the same recovery years.


In [ ]:
# Scale large storage values so axis offsets don't collide with titles
S_SCALE = 1e9   # → 10⁹ m³
DS_SCALE = 1e6  # → 10⁶ m³
LABEL_FS = 12
LEGEND_FS = 11
TITLE_FS = 13

ylabels = [
    "Total storage (10⁹ m³)",
    "Δ storage (10⁶ m³)",
    "Outlet flow (m³/h)",
    "Δ outlet flow (m³/h)",
]

fig, axes = plt.subplots(
    4,
    2,
    figsize=(10, 10),
    sharex="col",
    constrained_layout=True,
)
fig.set_constrained_layout_pads(h_pad=0.06, w_pad=0.04, hspace=0.08, wspace=0.04)


def recovery_window(ds, baseline, drought_length):
    t0 = SPINUP_YEARS + drought_length
    t1 = t0 + RECOVERY_YEARS
    d_sel = ds.sel(time=slice(t0, t1))
    b_sel = baseline.sel(time=slice(t0, t1))
    n = min(d_sel.sizes["time"], b_sel.sizes["time"])
    t = d_sel.time.values[:n] - t0
    return {
        "t": t,
        "S": d_sel.storage.values[:n] / S_SCALE,
        "Q": d_sel.outlet_flow.values[:n],
        "Sb": b_sel.storage.values[:n] / S_SCALE,
        "Qb": b_sel.outlet_flow.values[:n],
        "dS": (d_sel.storage.values[:n] - b_sel.storage.values[:n]) / DS_SCALE,
        "dQ": d_sel.outlet_flow.values[:n] - b_sel.outlet_flow.values[:n],
    }


for col, domain_name in enumerate(DOMAINS):
    baseline = series[domain_name]["short_baseline"]
    ax_S, ax_dS, ax_Q, ax_dQ = axes[:, col]

    for L in DROUGHT_LENGTHS:
        w = recovery_window(
            series[domain_name][f"{L}_year_drought"], baseline, L
        )
        ax_S.plot(w["t"], w["Sb"], color="0.75", lw=0.9, alpha=0.7)
        ax_Q.plot(w["t"], w["Qb"], color="0.75", lw=0.9, alpha=0.7)
        ax_S.plot(w["t"], w["S"], color=COLORS[L], lw=1.5, label=f"{L}-year")
        ax_Q.plot(w["t"], w["Q"], color=COLORS[L], lw=1.5, label=f"{L}-year")
        ax_dS.plot(w["t"], w["dS"], color=COLORS[L], lw=1.5, label=f"{L}-year")
        ax_dQ.plot(w["t"], w["dQ"], color=COLORS[L], lw=1.5, label=f"{L}-year")

    for ax in (ax_dS, ax_dQ):
        ax.axhline(0, color="k", lw=0.6)
    for ax in axes[:, col]:
        ax.axvline(0, color="k", ls="--", lw=0.8)
        ax.grid(True, alpha=0.3)
        ax.ticklabel_format(axis="y", style="plain", useOffset=False)

    ax_S.set_title(DOMAIN_LABELS[domain_name], fontsize=TITLE_FS)

# Potomac Δflow on the same absolute scale as outlet flow so the anomaly
# reads as small relative to the hydrograph magnitude (avoid auto-zoom on a spike).
q_lo, q_hi = axes[2, 0].get_ylim()
d_lo, d_hi = axes[3, 0].get_ylim()
axes[3, 0].set_ylim(min(q_lo, d_lo, 0.0), max(q_hi, d_hi))

for ax, label in zip(axes[:, 0], ylabels):
    ax.set_ylabel(label, fontsize=LABEL_FS)
fig.supxlabel("Years into recovery", fontsize=LABEL_FS)

handles = [
    Line2D([0], [0], color=COLORS[L], lw=1.5, label=f"{L}-year")
    for L in DROUGHT_LENGTHS
]
handles.append(Line2D([0], [0], color="0.65", lw=1.2, label="baseline"))
leg = fig.legend(
    handles=handles,
    loc="outside upper center",
    ncol=4,
    fontsize=LEGEND_FS,
    frameon=False,
)

out = FIG_DIR / "drought_recovery_totals_and_anomalies.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print("wrote", out)
plt.show()


## Fractional storage recovery

Storage anomaly normalized by the deficit at recovery onset ($\Delta S / \Delta S_{t=0}$).
1 = full onset deficit remaining; 0 = back to baseline.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=False)

for col, domain_name in enumerate(DOMAINS):
    ax = axes[col]
    baseline = series[domain_name]["short_baseline"]
    for L in DROUGHT_LENGTHS:
        ds = series[domain_name][f"{L}_year_drought"]
        t, dS, _ = recovery_anomaly(ds, baseline, L)
        denom = dS[0]
        if abs(denom) < 1e-6:
            print(f"warning: tiny onset deficit for {domain_name}/{L}-year")
            continue
        ax.plot(t, dS / denom, color=COLORS[L], lw=1.5, label=f"{L}-year")

    ax.axhline(0, color="k", lw=0.6)
    ax.axhline(1, color="0.5", ls=":", lw=0.8)
    ax.axvline(0, color="k", ls="--", lw=0.8)
    ax.grid(True, alpha=0.3)
    ax.set_title(DOMAIN_LABELS[domain_name], fontsize=13)
    ax.set_xlabel("Years into recovery", fontsize=12)
    ax.set_ylabel("Storage deficit remaining\n(fraction of onset deficit)", fontsize=12)
    ax.legend(fontsize=11)

fig.suptitle(
    "Fractional storage recovery (ΔS / ΔS at recovery onset)", y=1.02
)
plt.tight_layout()
out = FIG_DIR / "drought_recovery_by_length_fractional_storage.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print("wrote", out)
plt.show()


## 10-year drought: streamflow and storage

Last 3 spinup years → **10-year drought** → 5-year recovery.
Time axis is years from drought start ($t=0$ = onset; drought shaded $0$–$10$).
Columns are domains; rows are outlet streamflow and total storage vs `short_baseline`.
Baseline is shown on the storage panels only.


In [ ]:
from matplotlib.patches import Patch
from matplotlib.ticker import MultipleLocator

DROUGHT_L = 10
MEMBER = f"{DROUGHT_L}_year_drought"
SPINUP_KEEP = 3
PLOT_START = SPINUP_YEARS - SPINUP_KEEP  # 37
DROUGHT_END = SPINUP_YEARS + DROUGHT_L   # 50
PLOT_END = DROUGHT_END + RECOVERY_YEARS  # 55

S_SCALE = 1e9   # → 10⁹ m³
LABEL_FS = 12
LEGEND_FS = 11
TITLE_FS = 13
DROUGHT_COLOR = COLORS[DROUGHT_L]

ylabels = [
    "Outlet flow (m³/h)",
    "Total storage (10⁹ m³)",
]

fig, axes = plt.subplots(
    2,
    2,
    figsize=(10, 6.5),
    sharex="col",
    constrained_layout=True,
)
fig.set_constrained_layout_pads(h_pad=0.06, w_pad=0.04, hspace=0.08, wspace=0.04)

for col, domain_name in enumerate(DOMAINS):
    drought = series[domain_name][MEMBER].sel(time=slice(PLOT_START, PLOT_END))
    baseline = series[domain_name]["short_baseline"].sel(
        time=slice(PLOT_START, PLOT_END)
    )
    n = min(drought.sizes["time"], baseline.sizes["time"])
    t = drought.time.values[:n] - SPINUP_YEARS  # years from drought start
    q = drought.outlet_flow.values[:n]
    s = drought.storage.values[:n] / S_SCALE
    sb = baseline.storage.values[:n] / S_SCALE

    ax_q, ax_s = axes[:, col]
    ax_q.plot(t, q, color=DROUGHT_COLOR, lw=1.4, label=f"{DROUGHT_L}-year drought")
    ax_s.plot(t, sb, color="0.65", lw=1.0, label="baseline")
    ax_s.plot(t, s, color=DROUGHT_COLOR, lw=1.4, label=f"{DROUGHT_L}-year drought")

    for ax in axes[:, col]:
        ax.axvline(0, color="k", ls="--", lw=0.9)
        ax.axvline(DROUGHT_L, color="k", ls=":", lw=0.9)
        ax.axvspan(0, DROUGHT_L, color="C3", alpha=0.08)
        ax.grid(True, alpha=0.3)
        ax.ticklabel_format(axis="y", style="plain", useOffset=False)
        ax.xaxis.set_major_locator(MultipleLocator(2))
        ax.xaxis.set_minor_locator(MultipleLocator(1))

    ax_q.set_title(DOMAIN_LABELS[domain_name], fontsize=TITLE_FS)

for ax, label in zip(axes[:, 0], ylabels):
    ax.set_ylabel(label, fontsize=LABEL_FS)
fig.supxlabel("Year (from drought start)", fontsize=LABEL_FS)

handles = [
    Line2D([0], [0], color=DROUGHT_COLOR, lw=1.4, label=f"{DROUGHT_L}-year drought"),
    Line2D([0], [0], color="0.65", lw=1.2, label="baseline"),
    Patch(facecolor="C3", alpha=0.25, edgecolor="none", label="drought period"),
]
fig.legend(
    handles=handles,
    loc="outside upper center",
    ncol=3,
    fontsize=LEGEND_FS,
    frameon=False,
)

out = FIG_DIR / "ten_year_drought_streamflow_storage.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print("wrote", out)
plt.show()


## WTD anomaly maps — recovery start vs end

$\Delta$WTD = drought − `short_baseline` (m). **Positive = deeper water table** (drier).

Snapshots from condensed end-of-window fields:
- **Start:** last step of the final drought year (recovery onset)
- **End:** last step of the 5th recovery year

**Lime** cells mark the stream network (top 3% of late-spinup baseline overland flow).

In [ ]:
def condensed_year_path(domain_name, member, year_index):
    files = utils._file_locations(
        ENSEMBLE, member, domain_name, start_year=0, interval=INTERVAL
    )
    return files[year_index]


def _as_bool2d(active):
    """Accept xarray DataArray or ndarray."""
    if hasattr(active, "values"):
        active = active.values
    return np.asarray(active, dtype=bool)


def read_wtd_snapshot(domain_name, member, year_index, *, which="last"):
    """Masked WTD (y, x) at first or last condensed step of a sequence year."""
    path = condensed_year_path(domain_name, member, year_index)
    with xr.open_dataset(path) as ds:
        t_idx = 0 if which == "first" else -1
        wtd = ds["wtd"].isel(time=t_idx)
        mask = ds["mask"]
        if "z" in mask.dims:
            active = mask.any(dim="z") > 0
        else:
            active = mask > 0
        return wtd.where(active).load()


def wtd_anomaly_pair(domain_name, drought_length):
    """Return (start_anom, end_anom) DataArrays for one drought length."""
    member = f"{drought_length}_year_drought"
    start_year = SPINUP_YEARS + drought_length - 1  # end of drought
    end_year = SPINUP_YEARS + drought_length + RECOVERY_YEARS - 1

    d_start = read_wtd_snapshot(domain_name, member, start_year, which="last")
    b_start = read_wtd_snapshot(
        domain_name, "short_baseline", start_year, which="last"
    )
    d_end = read_wtd_snapshot(domain_name, member, end_year, which="last")
    b_end = read_wtd_snapshot(
        domain_name, "short_baseline", end_year, which="last"
    )
    return d_start - b_start, d_end - b_end


def baseline_mean_flow(domain_name, n_years=5, start_year=35):
    files = utils._file_locations(
        ENSEMBLE, "short_baseline", domain_name, 0, interval=INTERVAL
    )
    acc = None
    active = None
    for yi in range(start_year, start_year + n_years):
        with xr.open_dataset(files[yi]) as ds:
            flow = ds["overland_flow"].mean(dim="time")
            mask = ds["mask"]
            act = mask.any(dim="z") > 0 if "z" in mask.dims else mask > 0
            active = act if active is None else active
            acc = flow if acc is None else acc + flow
    return (acc / n_years).load(), active.load()


def stream_mask_from_flow(flow, active, percentile=STREAM_FLOW_PERCENTILE):
    active_b = _as_bool2d(active)
    vals = np.asarray(flow.where(active_b).values)
    thr = np.nanpercentile(vals[np.isfinite(vals)], percentile)
    streams = (np.asarray(flow.values) >= thr) & active_b
    return streams, float(thr)


def distance_to_stream_km(streams, active, dx_km=1.0):
    active_b = _as_bool2d(active)
    ny, nx = active_b.shape
    dist = np.full((ny, nx), np.nan, dtype=np.float64)
    q = deque()
    for y, x in zip(*np.where(streams & active_b)):
        dist[y, x] = 0.0
        q.append((y, x))
    while q:
        y, x = q.popleft()
        for dy, dx in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            nyi, nxi = y + dy, x + dx
            if (
                0 <= nyi < ny
                and 0 <= nxi < nx
                and active_b[nyi, nxi]
                and np.isnan(dist[nyi, nxi])
            ):
                dist[nyi, nxi] = dist[y, x] + 1.0
                q.append((nyi, nxi))
    return dist * dx_km


def elevation_from_slopes(sx, sy, active, outlet_xy, dx=1000.0, dy=1000.0):
    """Relative elevation (m) by integrating ParFlow slopes from the outlet."""
    import heapq

    active_b = _as_bool2d(active)
    ny, nx = active_b.shape
    ox, oy = outlet_xy
    elev = np.full((ny, nx), np.nan, dtype=np.float64)
    if not active_b[oy, ox]:
        yy, xx = np.where(active_b)
        i = int(np.argmin((xx - ox) ** 2 + (yy - oy) ** 2))
        ox, oy = int(xx[i]), int(yy[i])
    elev[oy, ox] = 0.0
    pq = [(0.0, oy, ox)]
    visited = np.zeros((ny, nx), dtype=bool)
    while pq:
        _, y, x = heapq.heappop(pq)
        if visited[y, x]:
            continue
        visited[y, x] = True
        z0 = elev[y, x]
        for dy_i, dx_i in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            nyi, nxi = y + dy_i, x + dx_i
            if not (0 <= nyi < ny and 0 <= nxi < nx):
                continue
            if not active_b[nyi, nxi] or visited[nyi, nxi]:
                continue
            if dx_i != 0:
                s = 0.5 * (sx[y, x] + sx[nyi, nxi])
                dz = s * (dx_i * dx)
            else:
                s = 0.5 * (sy[y, x] + sy[nyi, nxi])
                dz = s * (dy_i * dy)
            if np.isnan(elev[nyi, nxi]):
                elev[nyi, nxi] = z0 + dz
                heapq.heappush(pq, (abs(elev[nyi, nxi]), nyi, nxi))
    return elev - np.nanmin(elev)


def nearest_stream_indices(streams, active):
    active_b = _as_bool2d(active)
    ny, nx = active_b.shape
    iy = np.full((ny, nx), -1, dtype=np.int32)
    ix = np.full((ny, nx), -1, dtype=np.int32)
    q = deque()
    for y, x in zip(*np.where(streams & active_b)):
        iy[y, x] = y
        ix[y, x] = x
        q.append((y, x))
    while q:
        y, x = q.popleft()
        for dy, dx in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            nyi, nxi = y + dy, x + dx
            if (
                0 <= nyi < ny
                and 0 <= nxi < nx
                and active_b[nyi, nxi]
                and iy[nyi, nxi] < 0
            ):
                iy[nyi, nxi] = iy[y, x]
                ix[nyi, nxi] = ix[y, x]
                q.append((nyi, nxi))
    return iy, ix


def height_above_drainage(elev, streams, active):
    active_b = _as_bool2d(active)
    iy, ix = nearest_stream_indices(streams, active_b)
    hand = elev - elev[iy, ix]
    return np.where((iy >= 0) & active_b, hand, np.nan)


def spearman_rho(x, y):
    n = len(x)
    if n < 3:
        return np.nan, np.nan
    rx = np.argsort(np.argsort(x)).astype(np.float64)
    ry = np.argsort(np.argsort(y)).astype(np.float64)
    rx -= rx.mean()
    ry -= ry.mean()
    denom = np.sqrt((rx * rx).sum() * (ry * ry).sum())
    if denom == 0:
        return np.nan, np.nan
    rho = float((rx * ry).sum() / denom)
    if abs(rho) >= 1:
        return rho, 0.0
    z = 0.5 * np.log((1 + rho) / (1 - rho)) * np.sqrt(n - 3)
    return rho, erfc(abs(z) / sqrt(2))


wtd_anoms = {}
for domain_name in DOMAINS:
    wtd_anoms[domain_name] = {}
    for L in DROUGHT_LENGTHS:
        start, end = wtd_anomaly_pair(domain_name, L)
        wtd_anoms[domain_name][L] = {"start": start, "end": end}
        print(
            f"{domain_name} {L}-yr: start ΔWTD "
            f"[{float(start.min()):.2f}, {float(start.max()):.2f}]  "
            f"end ΔWTD [{float(end.min()):.2f}, {float(end.max()):.2f}]"
        )

landscape = {}
for domain_name in DOMAINS:
    flow, active_xr = baseline_mean_flow(domain_name)
    active = _as_bool2d(active_xr)
    streams, thr = stream_mask_from_flow(flow, active)
    with xr.open_dataset(condensed_year_path(domain_name, "short_baseline", 0)) as ds:
        sx = np.asarray(ds["slopex"].values)
        sy = np.asarray(ds["slopey"].values)
        if sx.ndim == 3:
            sx, sy = sx[0], sy[0]
        mask = ds["mask"]
        active = _as_bool2d(
            mask.any(dim="z") > 0 if "z" in mask.dims else mask > 0
        )
    elev = elevation_from_slopes(sx, sy, active, outlets[domain_name])
    landscape[domain_name] = {
        "active": active,
        "streams": streams,
        "stream_thr": thr,
        "dist": distance_to_stream_km(streams, active),
        "elev": elev,
        "hand": height_above_drainage(elev, streams, active),
        "slope_mag": np.where(active, np.sqrt(sx**2 + sy**2), np.nan),
    }
    print(
        f"{domain_name}: stream thr={thr:.4g}, "
        f"n_stream={int(streams.sum())}/{int(active.sum())}"
    )

In [ ]:
def _principal_axis_angle_deg(active):
    """Angle (deg CCW from +x) of the major axis of active cells."""
    yy, xx = np.where(active)
    if yy.size < 2:
        return 0.0
    pts = np.column_stack([xx.astype(np.float64), yy.astype(np.float64)])
    pts -= pts.mean(axis=0)
    cov = pts.T @ pts / max(pts.shape[0] - 1, 1)
    vals, vecs = np.linalg.eigh(cov)
    vx, vy = vecs[:, int(np.argmax(vals))]
    return float(np.degrees(np.arctan2(vy, vx)))


def _rotate2d(field, angle_deg, *, order=1):
    """Rotate 2D array by angle_deg (CCW) about center; reshape to fit.

    Pure NumPy (scipy.ndimage is broken in this env). order=0 nearest, 1 bilinear.
    """
    a = np.asarray(field, dtype=np.float64)
    ny, nx = a.shape
    theta = np.deg2rad(angle_deg)
    c, s = np.cos(theta), np.sin(theta)
    corners = np.array(
        [[-nx / 2, -ny / 2], [nx / 2, -ny / 2], [-nx / 2, ny / 2], [nx / 2, ny / 2]],
        dtype=np.float64,
    )
    R = np.array([[c, -s], [s, c]], dtype=np.float64)
    rot_corners = corners @ R.T
    min_xy = rot_corners.min(axis=0)
    max_xy = rot_corners.max(axis=0)
    out_nx = int(np.ceil(max_xy[0] - min_xy[0])) + 1
    out_ny = int(np.ceil(max_xy[1] - min_xy[1])) + 1

    yy, xx = np.meshgrid(np.arange(out_ny), np.arange(out_nx), indexing="ij")
    xd = xx + 0.5 - out_nx / 2
    yd = yy + 0.5 - out_ny / 2
    xs = c * xd + s * yd + nx / 2
    ys = -s * xd + c * yd + ny / 2

    out = np.full((out_ny, out_nx), np.nan, dtype=np.float64)
    if order == 0:
        xi = np.rint(xs - 0.5).astype(int)
        yi = np.rint(ys - 0.5).astype(int)
        ok = (xi >= 0) & (xi < nx) & (yi >= 0) & (yi < ny)
        out[ok] = a[yi[ok], xi[ok]]
        return out

    x0 = np.floor(xs - 0.5).astype(int)
    y0 = np.floor(ys - 0.5).astype(int)
    x1, y1 = x0 + 1, y0 + 1
    wx = (xs - 0.5) - x0
    wy = (ys - 0.5) - y0
    ok = (x0 >= 0) & (x1 < nx) & (y0 >= 0) & (y1 < ny)
    v00 = a[y0[ok], x0[ok]]
    v01 = a[y0[ok], x1[ok]]
    v10 = a[y1[ok], x0[ok]]
    v11 = a[y1[ok], x1[ok]]
    wxe, wye = wx[ok], wy[ok]
    val = (
        v00 * (1 - wxe) * (1 - wye)
        + v01 * wxe * (1 - wye)
        + v10 * (1 - wxe) * wye
        + v11 * wxe * wye
    )
    nanish = ~(np.isfinite(v00) & np.isfinite(v01) & np.isfinite(v10) & np.isfinite(v11))
    out[ok] = np.where(nanish, np.nan, val)
    return out


def _crop_to_valid(z, *others, pad=2):
    valid = np.isfinite(z)
    if not valid.any():
        return (z,) + others
    ys, xs = np.where(valid)
    y0 = max(int(ys.min()) - pad, 0)
    y1 = min(int(ys.max()) + pad + 1, z.shape[0])
    x0 = max(int(xs.min()) - pad, 0)
    x1 = min(int(xs.max()) + pad + 1, z.shape[1])
    sl = (slice(y0, y1), slice(x0, x1))
    return (z[sl],) + tuple(o[sl] for o in others)


def _prepare_map(anom_values, streams, active, *, align_landscape):
    z = np.where(active, np.asarray(anom_values, dtype=np.float64), np.nan)
    stream_grid = np.asarray(streams, dtype=bool)
    if align_landscape:
        angle = _principal_axis_angle_deg(active)
        z = _rotate2d(z, -angle, order=1)
        stream_r = _rotate2d(stream_grid.astype(np.float64), -angle, order=0)
        stream_grid = np.isfinite(stream_r) & (stream_r > 0.5)
        z, stream_grid = _crop_to_valid(z, stream_grid, pad=2)
    return z, stream_grid


def _plot_wtd_panel(ax, z, stream_grid, norm):
    ny, nx = z.shape
    im = ax.pcolormesh(
        np.arange(nx + 1),
        np.arange(ny + 1),
        z,
        shading="flat",
        cmap="RdBu_r",
        norm=norm,
    )
    sy, sx = np.where(stream_grid)
    if sy.size:
        ax.scatter(
            sx + 0.5, sy + 0.5, s=4, c="lime", marker="s", linewidths=0, zorder=3
        )
    ax.set_aspect("equal")
    # Make the axes box match the data aspect (removes letterboxing)
    ax.set_box_aspect(ny / nx)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlim(0, nx)
    ax.set_ylim(0, ny)
    for spine in ax.spines.values():
        spine.set_visible(False)
    return im


for domain_name in DOMAINS:
    sample = wtd_anoms[domain_name][DROUGHT_LENGTHS[0]]["start"].values
    align = sample.shape[0] > sample.shape[1]  # tall domains (potomac2)
    nrows = len(DROUGHT_LENGTHS)

    # Precompute one prepared map to size the figure from its aspect
    streams = landscape[domain_name]["streams"]
    active = landscape[domain_name]["active"]
    z0, _ = _prepare_map(
        wtd_anoms[domain_name][DROUGHT_LENGTHS[0]]["start"].values,
        streams,
        active,
        align_landscape=align,
    )
    map_aspect = z0.shape[1] / z0.shape[0]  # width / height

    # Grid: [row-label | start | end] ; short rows for wide maps
    label_w = 0.7
    map_w = 5.6 if align else 3.8
    fig_w = label_w + 2 * map_w + 0.85  # colorbar room
    # row height tracks map aspect closely to avoid empty vertical bands
    row_h = map_w / map_aspect + (0.45 if align else 0.55)
    fig_h = row_h * nrows + 0.45

    fig = plt.figure(figsize=(fig_w, fig_h))
    gs = fig.add_gridspec(
        nrows,
        3,
        width_ratios=[label_w, map_w, map_w],
        height_ratios=[1] * nrows,
        wspace=0.08,
        hspace=0.18 if align else 0.25,
        left=0.06,
        right=0.88,
        top=0.90,
        bottom=0.04,
    )

    vals = [
        wtd_anoms[domain_name][L][key].values[
            np.isfinite(wtd_anoms[domain_name][L][key].values)
        ]
        for L in DROUGHT_LENGTHS
        for key in ("start", "end")
    ]
    vmax = max(float(np.nanpercentile(np.abs(np.concatenate(vals)), 98)), 0.05)
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)

    im = None
    map_axes = []
    for row, L in enumerate(DROUGHT_LENGTHS):
        ax_lab = fig.add_subplot(gs[row, 0])
        ax_lab.axis("off")
        ax_lab.text(
            1.0,
            0.5,
            f"{L}-year",
            transform=ax_lab.transAxes,
            ha="right",
            va="center",
            fontsize=12,
            rotation=0,
        )

        for col, key in enumerate(("start", "end")):
            ax = fig.add_subplot(gs[row, col + 1])
            z, stream_grid = _prepare_map(
                wtd_anoms[domain_name][L][key].values,
                streams,
                active,
                align_landscape=align,
            )
            im = _plot_wtd_panel(ax, z, stream_grid, norm)
            map_axes.append(ax)
            if row == 0:
                ax.set_title(
                    "Recovery start (end of drought)"
                    if key == "start"
                    else "Recovery end (+5 yr average)",
                    fontsize=11,
                )

    cbar = fig.colorbar(im, ax=map_axes, fraction=0.025, pad=0.02)
    cbar.set_label("Δ WTD (m)  (+ deeper / drier)", fontsize=12)
    map_axes[1].legend(
        handles=[
            Line2D(
                [0],
                [0],
                marker="s",
                color="w",
                markerfacecolor="lime",
                markersize=8,
                label="stream (top 3% baseline flow)",
            )
        ],
        loc="upper right",
        fontsize=10,
        framealpha=0.9,
    )
    fig.suptitle(
        f"{DOMAIN_LABELS[domain_name]}: WTD anomaly vs short_baseline (219 h; streams overlaid)",
        fontsize=12,
    )
    out = FIG_DIR / f"{domain_name}_drought_recovery_wtd_anomaly_maps.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    print("wrote", out, f"(align_landscape={align}, map_aspect={map_aspect:.2f})")
    plt.show()


## Where do persistent WTD anomalies live?

Covariates (no DEM on disk):
- **Distance to stream** — BFS distance to cells in the top 3% of late-spinup baseline overland flow
- **Relative elevation / HAND** — elevation reconstructed by integrating `slopex`/`slopey` from the outlet; HAND = elev − elev at nearest stream
- **Slope magnitude** — $\sqrt{s_x^2 + s_y^2}$

Focus: end-of-recovery ΔWTD for the **10-year** drought (most persistent).

In [ ]:
for domain_name in DOMAINS:
    land = landscape[domain_name]
    nonstream = land["active"] & (~land["streams"])
    print(f"\n=== {domain_name} Spearman(end ΔWTD, covariate) ===")
    for L in DROUGHT_LENGTHS:
        end = wtd_anoms[domain_name][L]["end"].values
        m = nonstream & np.isfinite(end)
        for name, arr in [
            ("dist_to_stream_km", land["dist"]),
            ("rel_elev_m", land["elev"]),
            ("HAND_m", land["hand"]),
            ("slope_mag", land["slope_mag"]),
        ]:
            mm = m & np.isfinite(arr)
            rho, p = spearman_rho(arr[mm], end[mm])
            print(f"  {L}-yr vs {name}: rho={rho:+.3f}  p={p:.2e}  n={mm.sum()}")

    end = wtd_anoms[domain_name][10]["end"].values
    m = nonstream & np.isfinite(end)
    thr = np.nanpercentile(end[m], 80)
    persistent = m & (end >= thr)
    other = m & (end < thr)
    print(
        f"{domain_name} 10-yr persistent = top 20% end ΔWTD "
        f"(≥ {thr:.3f} m), n={persistent.sum()}"
    )
    for name, arr in [
        ("dist_to_stream_km", land["dist"]),
        ("rel_elev_m", land["elev"]),
        ("HAND_m", land["hand"]),
        ("slope_mag", land["slope_mag"]),
    ]:
        print(
            f"  {name}: persistent median={np.nanmedian(arr[persistent]):.4g}  "
            f"other median={np.nanmedian(arr[other]):.4g}"
        )

fig, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)
covars = [
    ("dist", "Distance to stream (km)"),
    ("hand", "Height above drainage (m)"),
    ("slope_mag", "Slope magnitude (-)"),
]
for row, domain_name in enumerate(DOMAINS):
    land = landscape[domain_name]
    end = wtd_anoms[domain_name][10]["end"].values
    # clip rare multi-meter outliers for display
    m = land["active"] & (~land["streams"]) & np.isfinite(end) & (end < 2.0)
    for col, (key, label) in enumerate(covars):
        ax = axes[row, col]
        x = land[key]
        mm = m & np.isfinite(x)
        ax.hexbin(x[mm], end[mm], gridsize=35, cmap="viridis", mincnt=1, bins="log")
        qs = np.nanpercentile(x[mm], np.linspace(5, 95, 12))
        centers, meds = [], []
        for a, b in zip(qs[:-1], qs[1:]):
            sel = mm & (x >= a) & (x < b)
            if sel.sum() < 5:
                continue
            centers.append(0.5 * (a + b))
            meds.append(np.nanmedian(end[sel]))
        ax.plot(centers, meds, "r-o", ms=3, lw=1.5, label="bin median")
        ax.set_xlabel(label)
        if col == 0:
            ax.set_ylabel(f"{DOMAIN_LABELS[domain_name]}\nend ΔWTD (m) — 10-yr", fontsize=12)
        if row == 0 and col == 0:
            ax.legend(fontsize=10)

fig.suptitle(
    "End-of-recovery WTD anomaly (10-yr drought) vs landscape covariates",
    fontsize=11,
)
out = FIG_DIR / "drought_recovery_wtd_persistence_vs_covariates.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print("wrote", out)
plt.show()
